phases/11-llm-engineering/11-caching-cost

In [ ]:
import sys, json, types
lrn_llm = types.ModuleType("lrn_llm")
try:
    from pyodide.http import pyfetch as _pyfetch
    _IN_PYODIDE = True
except ImportError:
    import urllib.request as _urlreq
    _IN_PYODIDE = False
lrn_llm.API_BASE = "/api/llm"  # same-origin proxy; server injects the gateway key
lrn_llm.DEFAULT_MODEL = "azure/gpt-5.4-mini"
lrn_llm.API_KEY = ""  # optional; set in Step 0a

async def _lrn_call(messages, *, system=None, max_tokens=400, model=None):
    if system is not None:
        messages = [{"role": "system", "content": system}] + list(messages)
    payload = {"model": model or lrn_llm.DEFAULT_MODEL, "messages": messages,
               "max_completion_tokens": max_tokens}
    headers = {"content-type": "application/json"}
    _key = lrn_llm.API_KEY
    if _key:
        headers["Authorization"] = "Bearer " + _key
    url = lrn_llm.API_BASE.rstrip("/") + "/chat/completions"
    body = json.dumps(payload)
    if _IN_PYODIDE:
        r = await _pyfetch(url, method="POST", headers=headers, body=body)
        data = await r.json()
    else:
        req = _urlreq.Request(url, method="POST", headers=headers, data=body.encode("utf-8"))
        with _urlreq.urlopen(req, timeout=60) as r:
            data = json.loads(r.read())
    if "error" in data:
        raise RuntimeError("LLM error: " + str(data["error"]))
    return data

def _lrn_text(r):
    ch = (r or {}).get("choices") or []
    return (ch[0].get("message", {}) or {}).get("content", "") if ch else ""

async def _lrn_ping():
    r = await _lrn_call([{"role": "user", "content": "Reply with exactly: OK"}], max_tokens=5)
    return {"ok": _lrn_text(r).strip().upper().startswith("OK"), "model": r.get("model")}

lrn_llm.call = _lrn_call
lrn_llm.text = _lrn_text
lrn_llm.ping = _lrn_ping
print("✅ notebook ready · endpoint:", lrn_llm.API_BASE)

## Step 0a — Endpoint & Key

In [ ]:
lrn_llm.API_KEY = ""
print(f"🔧 Endpoint: {lrn_llm.API_BASE}")
print(f"📦 Default model: {lrn_llm.DEFAULT_MODEL}")
print(f"🔑 API key: {'set' if lrn_llm.API_KEY else 'not set (gateway auth)'}") 

## Step 1 — Reachability

In [ ]:
r = await lrn_llm.ping()
print(f"✅ LLM erreichbar: {r}")

## Step 2 — Cost Calculator

Every API call has a cost. Let's implement a token cost calculator that knows current pricing for the major models (GPT-4o, Claude Sonnet, etc). This is the foundation of all cost optimization.

In [ ]:
import hashlib, time, math
from dataclasses import dataclass

MODEL_PRICING = {
    "azure/gpt-5.4-mini": {"input": 0.30, "output": 0.90, "cached_input": 0.15},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60, "cached_input": 0.075},
    "claude-opus-4": {"input": 15.00, "output": 75.00, "cached_input": 1.50},
    "claude-sonnet-4": {"input": 3.00, "output": 15.00, "cached_input": 0.30},
    "claude-haiku-3.5": {"input": 0.80, "output": 4.00, "cached_input": 0.08},
}

def calculate_cost(model, input_tokens, output_tokens, cached_input_tokens=0):
    # Gateway model ids carry a provider prefix (e.g. "azure/gpt-5.4-mini"). Strip it off for pricing lookup.;
    # normalize to the bare model name used in MODEL_PRICING.
    pricing_key = model.split("/")[-1]
    if pricing_key not in MODEL_PRICING:
        return {"error": f"Unknown model: {model}"}
    pricing = MODEL_PRICING[pricing_key]
    non_cached = input_tokens - cached_input_tokens
    input_cost = (non_cached / 1_000_000) * pricing["input"]
    cached_cost = (cached_input_tokens / 1_000_000) * pricing["cached_input"]
    output_cost = (output_tokens / 1_000_000) * pricing["output"]
    total = input_cost + cached_cost + output_cost
    return {
        "model": model,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cached_input_tokens": cached_input_tokens,
        "input_cost": round(input_cost, 6),
        "cached_input_cost": round(cached_cost, 6),
        "output_cost": round(output_cost, 6),
        "total_cost": round(total, 6),
    }

print("💰 Cost Calculator Ready")
print("\n--- Example: 1000 input + 500 output tokens per model ---")
for model in ["gpt-4o", "gpt-4o-mini", "claude-sonnet-4", "claude-haiku-3.5"]:
    cost = calculate_cost(model, 1000, 500)
    print(f"{model:20} ${cost['total_cost']:.6f}")

## Step 3 — Exact Cache

For identical queries (temperature=0), hash the full prompt and return cached responses instantly. This saves 100% cost on cache hits.

In [ ]:
class ExactCache:
    def __init__(self, max_size=1000, ttl_seconds=3600):
        self.cache = {}
        self.max_size = max_size
        self.ttl = ttl_seconds
        self.hits = 0
        self.misses = 0

    def _hash(self, model, messages, temperature):
        key_data = json.dumps({"model": model, "messages": messages, "temperature": temperature}, sort_keys=True)
        return hashlib.sha256(key_data.encode()).hexdigest()

    def get(self, model, messages, temperature=0.0):
        if temperature > 0:
            self.misses += 1
            return None
        key = self._hash(model, messages, temperature)
        if key in self.cache:
            entry = self.cache[key]
            if time.time() - entry["timestamp"] < self.ttl:
                self.hits += 1
                entry["access_count"] += 1
                return entry["response"]
            del self.cache[key]
        self.misses += 1
        return None

    def put(self, model, messages, temperature, response):
        if temperature > 0:
            return
        if len(self.cache) >= self.max_size:
            oldest_key = min(self.cache, key=lambda k: self.cache[k]["timestamp"])
            del self.cache[oldest_key]
        key = self._hash(model, messages, temperature)
        self.cache[key] = {"response": response, "timestamp": time.time(), "access_count": 1}

    def stats(self):
        total = self.hits + self.misses
        return {"hits": self.hits, "misses": self.misses, "hit_rate": round(self.hits / total, 4) if total > 0 else 0, "size": len(self.cache)}

exact = ExactCache()
msgs = [{"role": "user", "content": "What is the return policy?"}]
print("🔍 Exact Cache Demo")
print(f"First lookup (MISS): {exact.get('gpt-4o-mini', msgs, 0.0)}")
exact.put('gpt-4o-mini', msgs, 0.0, "You can return items within 30 days.")
result = exact.get('gpt-4o-mini', msgs, 0.0)
print(f"Second lookup (HIT): {result}")
print(f"Stats: {exact.stats()}")

## Step 4 — Semantic Cache

Similar queries should get the same answer. Embed queries and find cached responses when similarity exceeds a threshold (e.g., "What is the return policy?" vs "How do I return an item?").

In [ ]:
def simple_embed(text):
    """Bag-of-words embedding (cosine similarity ready)."""
    words = text.lower().split()
    vocab = {}
    for w in words:
        vocab[w] = vocab.get(w, 0) + 1
    norm = math.sqrt(sum(v * v for v in vocab.values()))
    return {k: v / norm for k, v in vocab.items()} if norm > 0 else {}

def cosine_similarity(a, b):
    """Dot product of normalized vectors."""
    if not a or not b:
        return 0.0
    all_keys = set(a) | set(b)
    return sum(a.get(k, 0) * b.get(k, 0) for k in all_keys)

class SemanticCache:
    def __init__(self, similarity_threshold=0.85, max_size=500, ttl_seconds=3600):
        self.entries = []
        self.threshold = similarity_threshold
        self.max_size = max_size
        self.ttl = ttl_seconds
        self.hits = 0
        self.misses = 0

    def get(self, query):
        query_embedding = simple_embed(query)
        now = time.time()
        best_match = None
        best_sim = 0.0
        for entry in self.entries:
            if now - entry["timestamp"] > self.ttl:
                continue
            sim = cosine_similarity(query_embedding, entry["embedding"])
            if sim > best_sim:
                best_sim = sim
                best_match = entry
        if best_match and best_sim >= self.threshold:
            self.hits += 1
            best_match["access_count"] += 1
            return {"response": best_match["response"], "similarity": round(best_sim, 4), "original_query": best_match["query"]}
        self.misses += 1
        return None

    def put(self, query, response):
        if len(self.entries) >= self.max_size:
            self.entries.sort(key=lambda e: e["timestamp"])
            self.entries.pop(0)
        self.entries.append({"query": query, "embedding": simple_embed(query), "response": response, "timestamp": time.time(), "access_count": 1})

    def stats(self):
        total = self.hits + self.misses
        return {"hits": self.hits, "misses": self.misses, "hit_rate": round(self.hits / total, 4) if total > 0 else 0, "size": len(self.entries)}

sem_cache = SemanticCache(similarity_threshold=0.75)
print("🧠 Semantic Cache Demo")
test_queries = [
    ("What is the return policy?", "Items can be returned within 30 days."),
    ("How do I return an item?", None),
    ("What are your store hours?", "Open 9am-9pm Mon-Sat."),
    ("When does the store open?", None),
]
for query, response in test_queries:
    cached = sem_cache.get(query)
    if cached:
        print(f"✓ '{query}' -> HIT (sim={cached['similarity']}, orig='{cached['original_query']}')")
    elif response:
        sem_cache.put(query, response)
        print(f"✗ '{query}' -> MISS (stored)")
    else:
        print(f"✗ '{query}' -> MISS (no match)")
print(f"\nStats: {sem_cache.stats()}")

## Step 5 — Rate Limiting

Token bucket algorithm: each user has a bucket that refills at a fixed rate. Requests consume tokens. When empty, requests are blocked. Protects your budget from runaway costs.

In [ ]:
class TokenBucketRateLimiter:
    def __init__(self):
        self.buckets = {}
        self.tiers = {
            "free": {"capacity": 50_000, "refill_rate": 500, "max_rpm": 10},
            "pro": {"capacity": 500_000, "refill_rate": 5_000, "max_rpm": 60},
            "enterprise": {"capacity": 5_000_000, "refill_rate": 50_000, "max_rpm": 300},
        }

    def _get_bucket(self, user_id, tier="free"):
        if user_id not in self.buckets:
            cfg = self.tiers.get(tier, self.tiers["free"])
            self.buckets[user_id] = {
                "tokens": cfg["capacity"],
                "capacity": cfg["capacity"],
                "refill_rate": cfg["refill_rate"],
                "last_refill": time.time(),
                "max_rpm": cfg["max_rpm"],
                "tier": tier,
                "total_used": 0,
            }
        return self.buckets[user_id]

    def _refill(self, bucket):
        now = time.time()
        elapsed = now - bucket["last_refill"]
        refill = int(elapsed * bucket["refill_rate"])
        if refill > 0:
            bucket["tokens"] = min(bucket["capacity"], bucket["tokens"] + refill)
            bucket["last_refill"] = now

    def check(self, user_id, tokens_needed, tier="free"):
        bucket = self._get_bucket(user_id, tier)
        self._refill(bucket)
        if bucket["tokens"] < tokens_needed:
            deficit = tokens_needed - bucket["tokens"]
            wait = deficit / bucket["refill_rate"]
            return {"allowed": False, "reason": "insufficient_tokens", "available": bucket["tokens"], "wait_sec": round(wait, 1)}
        return {"allowed": True, "available": bucket["tokens"]}

    def consume(self, user_id, tokens_used, tier="free"):
        bucket = self._get_bucket(user_id, tier)
        bucket["tokens"] -= tokens_used
        bucket["total_used"] += tokens_used

    def get_usage(self, user_id):
        if user_id not in self.buckets:
            return {"error": "User not found"}
        b = self.buckets[user_id]
        return {"tier": b["tier"], "remaining": b["tokens"], "capacity": b["capacity"], "used": b["total_used"]}

limiter = TokenBucketRateLimiter()
print("🚦 Token Bucket Rate Limiter Demo (free tier: 50K tokens)")
for i in range(12):
    check = limiter.check("user_1", 10000, "free")
    if check["allowed"]:
        limiter.consume("user_1", 10000, "free")
        status = "✓ ALLOWED"
    else:
        status = f"✗ BLOCKED ({check['reason']}, wait {check['wait_sec']}s)"
    if i < 5 or not check["allowed"]:
        print(f"  Request {i+1}: {status}")
print(f"\nUsage: {limiter.get_usage('user_1')}")

## Step 6 — Real LLM Call with Caching

Now let's make REAL API calls to the LLM and demonstrate caching in action. We'll ask a customer service question, cache it, then ask a similar question and catch the cache hit.

In [ ]:
import time as t

SYSTEM_PROMPT = """You are a helpful customer service agent for an online retailer. 
Answer questions about returns, shipping, and policies briefly (2-3 sentences max)."""

print("📞 Real LLM Calls with Caching Demo\n")
print("=" * 60)
print("Call 1: Original query (cache MISS)")
print("=" * 60)

start = t.time()
resp1 = await lrn_llm.call(
    [{"role": "user", "content": "What is the return policy?"}],
    system=SYSTEM_PROMPT,
    max_tokens=150
)
latency1 = t.time() - start
text1 = lrn_llm.text(resp1)

print(f"Response: {text1}")
print(f"Latency: {latency1*1000:.0f}ms")
print(f"Input tokens: {resp1.get('usage', {}).get('prompt_tokens', 0)}")
print(f"Output tokens: {resp1.get('usage', {}).get('completion_tokens', 0)}")

# Cache the response
exact.put('gpt-4o-mini', [{"role": "user", "content": "What is the return policy?"}], 0.0, text1)
print(f"✓ Cached response for next identical query\n")

print("=" * 60)
print("Call 2: Identical query (cache HIT - no LLM call)")
print("=" * 60)

start = t.time()
messages2 = [{"role": "user", "content": "What is the return policy?"}]
cached = exact.get('gpt-4o-mini', messages2, 0.0)
latency2 = t.time() - start

if cached:
    print(f"Response: {cached} (from cache)")
    print(f"Latency: {latency2*1000:.1f}ms")
    print(f"✓ Cache HIT: {latency1*1000/latency2:.0f}x faster, 100% cost savings")
else:
    print("Cache miss (unexpected)")

## Step 7 — Cost Tracking & Budget Alerts

Log every API call with cost and token usage. Detect budget overruns before they happen.

In [ ]:
class CostTracker:
    def __init__(self, monthly_budget=100.0):
        self.logs = []
        self.monthly_budget = monthly_budget
        self.alerts = []

    def log_call(self, model, input_tokens, output_tokens, cached_input_tokens=0, latency_ms=0, user_id="anon", cache_status="miss"):
        cost = calculate_cost(model, input_tokens, output_tokens, cached_input_tokens)
        entry = {
            "timestamp": time.time(),
            "model": model,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "latency_ms": latency_ms,
            "cost": cost["total_cost"],
            "user_id": user_id,
            "cache_status": cache_status,
        }
        self.logs.append(entry)
        self._check_budget()
        return entry

    def _check_budget(self):
        total = self.total_cost()
        pct = total / self.monthly_budget if self.monthly_budget > 0 else 0
        if pct >= 0.95 and not any(a["level"] == "stop" for a in self.alerts):
            self.alerts.append({"level": "stop", "msg": f"95% budget consumed: ${total:.2f}"})
        elif pct >= 0.85 and not any(a["level"] == "throttle" for a in self.alerts):
            self.alerts.append({"level": "throttle", "msg": f"85% budget consumed: ${total:.2f}"})
        elif pct >= 0.70 and not any(a["level"] == "warning" for a in self.alerts):
            self.alerts.append({"level": "warning", "msg": f"70% budget consumed: ${total:.2f}"})

    def total_cost(self):
        return round(sum(e["cost"] for e in self.logs), 6)

    def cost_by_model(self):
        by_model = {}
        for e in self.logs:
            m = e["model"]
            if m not in by_model:
                by_model[m] = {"calls": 0, "cost": 0, "in_tokens": 0, "out_tokens": 0}
            by_model[m]["calls"] += 1
            by_model[m]["cost"] = round(by_model[m]["cost"] + e["cost"], 6)
            by_model[m]["in_tokens"] += e["input_tokens"]
            by_model[m]["out_tokens"] += e["output_tokens"]
        return by_model

    def cache_savings(self):
        cache_hits = [e for e in self.logs if e["cache_status"] == "hit"]
        if not cache_hits:
            return {"saved": 0, "hits": 0}
        saved = sum(calculate_cost(e["model"], e["input_tokens"], e["output_tokens"])["total_cost"] for e in cache_hits)
        return {"saved": round(saved, 4), "hits": len(cache_hits)}

    def summary(self):
        if not self.logs:
            return {"calls": 0, "cost": 0}
        cache_hits = sum(1 for e in self.logs if e["cache_status"] == "hit")
        return {
            "calls": len(self.logs),
            "cost": self.total_cost(),
            "avg_cost_per_call": round(self.total_cost() / len(self.logs), 6),
            "cache_hit_rate": round(cache_hits / len(self.logs), 4),
            "by_model": self.cost_by_model(),
            "savings": self.cache_savings(),
            "alerts": self.alerts,
        }

tracker = CostTracker(monthly_budget=50.0)
print("💾 Cost Tracker Demo\n")

# Log the real LLM call from Step 6
tracker.log_call(
    "azure/gpt-5.4-mini",
    resp1.get('usage', {}).get('prompt_tokens', 100),
    resp1.get('usage', {}).get('completion_tokens', 50),
    latency_ms=latency1*1000,
    user_id="user_1",
    cache_status="miss"
)

# Log the cache hit
tracker.log_call(
    "azure/gpt-5.4-mini",
    100, 50, latency_ms=1, user_id="user_1", cache_status="hit"
)

summary = tracker.summary()
print(f"Total calls: {summary['calls']}")
print(f"Total cost: ${summary['cost']:.6f} / ${tracker.monthly_budget}")
print(f"Cache hit rate: {summary['cache_hit_rate']:.0%}")
print(f"Cache savings: ${summary['savings']['saved']:.6f}")
if summary['alerts']:
    for alert in summary['alerts']:
        print(f"⚠️  [{alert['level'].upper()}] {alert['msg']}")

## Step 8 — Model Routing

Not every query needs an expensive model. Route simple queries (FAQ, lookups) to cheap models (GPT-4o-mini, Haiku) and complex queries (reasoning, code) to expensive models (GPT-4o, Claude Opus).

In [ ]:
SIMPLE_KEYWORDS = ["what time", "hours", "address", "phone", "price", "return", "hello", "hi"]
COMPLEX_KEYWORDS = ["analyze", "compare", "explain why", "code", "debug", "design"]

def classify_complexity(query):
    q = query.lower()
    if len(q.split()) <= 5 or any(kw in q for kw in SIMPLE_KEYWORDS):
        return "simple"
    if any(kw in q for kw in COMPLEX_KEYWORDS):
        return "complex"
    return "medium"

def route_model(query, tier="pro"):
    complexity = classify_complexity(query)
    routing = {
        "simple": {"free": "gpt-4o-mini", "pro": "gpt-4o-mini", "ent": "gpt-4o-mini"},
        "medium": {"free": "gpt-4o-mini", "pro": "claude-sonnet-4", "ent": "claude-sonnet-4"},
        "complex": {"free": "gpt-4o-mini", "pro": "gpt-4o", "ent": "claude-opus-4"},
    }
    model = routing[complexity].get(tier, "gpt-4o-mini")
    return {"query": query, "complexity": complexity, "model": model}

print("🎯 Model Routing Demo (Pro tier)\n")
test_queries = [
    "What time do you close?",
    "Analyze the cost breakdown",
    "Hello",
    "Write code for binary search",
    "What is your address?",
]
for q in test_queries:
    route = route_model(q, "pro")
    cost_expensive = calculate_cost("gpt-4o", 800, 200)
    cost_routed = calculate_cost(route["model"], 800, 200)
    savings = cost_expensive["total_cost"] - cost_routed["total_cost"]
    print(f"{q:35} → {route['model']:18} save ${savings:.6f} per call")

## Step 9 — Full Pipeline Simulation

Before-and-after: a RAG chatbot with 10 common queries. One path uses no optimization (single expensive model), the other uses all techniques (caching + routing). Watch the cost drop.

In [ ]:
def simulate_llm_call(model, query):
    """Simulate token usage for a query."""
    in_tokens = len(query.split()) * 4 + 500
    out_tokens = 150 + (len(query.split()) * 2)
    return {"model": model, "input_tokens": in_tokens, "output_tokens": out_tokens, "latency_ms": 200 + out_tokens * 2}

queries = [
    "What is the return policy?",
    "How do I return an item?",
    "What are your store hours?",
    "When do you open?",
    "Explain the supply chain",
    "Tell me about inventory",
    "Hello",
    "What is your phone?",
    "Design a shipping strategy",
    "Analyze demand patterns",
]

print("\n" + "=" * 60)
print("Before Optimization: single model (gpt-4o), no caching")
print("=" * 60)

before = CostTracker(monthly_budget=1000)
for q in queries:
    result = simulate_llm_call("gpt-4o", q)
    before.log_call("gpt-4o", result["input_tokens"], result["output_tokens"], latency_ms=result["latency_ms"], cache_status="miss")

before_summary = before.summary()
print(f"Total cost: ${before_summary['cost']:.6f}")
print(f"Avg cost/call: ${before_summary['avg_cost_per_call']:.6f}")
print(f"Avg latency: {before_summary.get('avg_latency_ms', 'N/A')}ms")

print("\n" + "=" * 60)
print("After Optimization: caching + routing")
print("=" * 60)

after = CostTracker(monthly_budget=1000)
exact_opt = ExactCache()
sem_opt = SemanticCache(similarity_threshold=0.75)

for q in queries:
    messages = [{"role": "user", "content": q}]
    cached = exact_opt.get("gpt-4o-mini", messages, 0.0)
    if cached:
        after.log_call("gpt-4o-mini", 0, 0, latency_ms=5, cache_status="hit")
        continue
    
    sem_cached = sem_opt.get(q)
    if sem_cached:
        after.log_call("gpt-4o-mini", 0, 0, latency_ms=15, cache_status="hit")
        continue
    
    route = route_model(q)
    result = simulate_llm_call(route["model"], q)
    after.log_call(route["model"], result["input_tokens"], result["output_tokens"], latency_ms=result["latency_ms"], cache_status="miss")
    exact_opt.put(route["model"], messages, 0.0, f"Response to {q}")
    sem_opt.put(q, f"Response to {q}")

after_summary = after.summary()
print(f"Total cost: ${after_summary['cost']:.6f}")
print(f"Avg cost/call: ${after_summary['avg_cost_per_call']:.6f}")
print(f"Cache hit rate: {after_summary['cache_hit_rate']:.0%}")
print(f"Cache savings: ${after_summary['savings']['saved']:.6f}")

if before_summary['cost'] > 0:
    savings_pct = (1 - after_summary['cost'] / before_summary['cost']) * 100
    print(f"\n🎉 Total savings: {savings_pct:.0f}% cost reduction")
    print(f"   ${before_summary['cost']:.6f} → ${after_summary['cost']:.6f}")

## Step 10 — Try It Yourself

Make a REAL LLM call with your own prompt. Experiment with different system instructions, track the cost, and see how caching works.

In [ ]:
# TODO: Customize this query and system prompt
# Try asking different questions and see how cost changes
# Experiment with similar questions to trigger semantic cache hits

YOUR_SYSTEM = """You are a helpful assistant. Be concise (1-2 sentences)."""
YOUR_QUERY = "What are the benefits of machine learning?"

print(f"🚀 Your Custom LLM Call\n")
print(f"System: {YOUR_SYSTEM}")
print(f"Query: {YOUR_QUERY}\n")

start = t.time()
resp = await lrn_llm.call(
    [{"role": "user", "content": YOUR_QUERY}],
    system=YOUR_SYSTEM,
    max_tokens=200
)
latency = t.time() - start
text = lrn_llm.text(resp)

print(f"Response:\n{text}\n")

in_tokens = resp.get('usage', {}).get('prompt_tokens', 100)
out_tokens = resp.get('usage', {}).get('completion_tokens', 50)
cost = calculate_cost("azure/gpt-5.4-mini", in_tokens, out_tokens)

print(f"📊 Cost Breakdown:")
print(f"  Input tokens: {in_tokens}")
print(f"  Output tokens: {out_tokens}")
print(f"  Cost: ${cost['total_cost']:.6f}")
print(f"  Latency: {latency*1000:.0f}ms")
print(f"\n💡 Try asking a similar question next to see semantic caching in action!")